In [1]:
%pip install faster-whisper transformers opensmile sounddevice numpy torch torchaudio silero-vad

     ---------------------------------------- 0.0/9.1 MB ? eta -:--:--
     --- ------------------------------------ 0.8/9.1 MB 26.2 MB/s eta 0:00:01
     -------- ------------------------------- 2.0/9.1 MB 30.9 MB/s eta 0:00:01
     -------------- ------------------------- 3.4/9.1 MB 30.7 MB/s eta 0:00:01
     -------------------- ------------------- 4.8/9.1 MB 27.7 MB/s eta 0:00:01
     ---------------------------- ----------- 6.5/9.1 MB 29.5 MB/s eta 0:00:01
     ----------------------------------- ---- 8.0/9.1 MB 28.4 MB/s eta 0:00:01
     ---------------------------------------  9.1/9.1 MB 30.8 MB/s eta 0:00:01
     ---------------------------------------  9.1/9.1 MB 30.8 MB/s eta 0:00:01
     ---------------------------------------- 9.1/9.1 MB 24.4 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import queue
import threading
import time
import os

import numpy as np
import sounddevice as sd

from faster_whisper import WhisperModel

import opensmile

from transformers import pipeline
from silero_vad import (
    load_silero_vad,
    get_speech_timestamps
)

c:\Users\igorc\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
SAMPLE_RATE = 16000
CHANNELS = 1

# tempo de cada chunk de áudio
CHUNK_DURATION = 15  # segundos

# pasta temporária
TEMP_DIR = "temp_audio"

os.makedirs(TEMP_DIR, exist_ok=True)

In [4]:
audio_queue = queue.Queue()

vad_model = load_silero_vad()

In [5]:
print("Carregando Whisper...")

whisper_model = WhisperModel(
    "small",
    device="cpu",      # troque para "cpu" se necessário
    compute_type="int8"
)

print("Whisper carregado.")

Carregando Whisper...
Whisper carregado.


In [6]:
print("Carregando modelo de emoção...")

emotion_classifier = pipeline(
    "audio-classification",
    model="ehcalabres/wav2vec2-lg-xlsr-en-speech-emotion-recognition"
)

print("SpeechBrain carregado.")

Carregando modelo de emoção...


Loading weights: 100%|██████████| 422/422 [00:00<00:00, 1599.34it/s]
[transformers] Wav2Vec2ForSequenceClassification LOAD REPORT from: ehcalabres/wav2vec2-lg-xlsr-en-speech-emotion-recognition
Key                      | Status     | 
-------------------------+------------+-
classifier.output.bias   | UNEXPECTED | 
classifier.output.weight | UNEXPECTED | 
classifier.dense.bias    | UNEXPECTED | 
classifier.dense.weight  | UNEXPECTED | 
projector.weight         | MISSING    | 
classifier.weight        | MISSING    | 
classifier.bias          | MISSING    | 
projector.bias           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


SpeechBrain carregado.


In [7]:

print("Carregando openSMILE...")

smile = opensmile.Smile(
    feature_set=opensmile.FeatureSet.eGeMAPSv02,
    feature_level=opensmile.FeatureLevel.Functionals,
)

print("openSMILE carregado.")

Carregando openSMILE...
openSMILE carregado.


In [8]:
recorded_chunks = []

last_save_time = time.time()


def audio_callback(indata, frames, time_info, status):

    global recorded_chunks
    global last_save_time

    if status:
        print(status)

    recorded_chunks.append(indata.copy())

    elapsed = time.time() - last_save_time

    if elapsed >= CHUNK_DURATION:

        audio_data = np.concatenate(recorded_chunks, axis=0)

        audio_queue.put(audio_data)

        recorded_chunks = []

        last_save_time = time.time()

# ============================================================
# PROCESSAMENTO
# ============================================================

In [9]:
def process_audio():

    while True:

        audio_data = audio_queue.get()

        try:

            # ==========================================
            # NORMALIZA ÁUDIO
            # ==========================================

            audio_np = audio_data.flatten()

            # ==========================================
            # VAD - DETECÇÃO DE FALA
            # ==========================================

            speech_timestamps = get_speech_timestamps(
                audio_np,
                vad_model,
                sampling_rate=SAMPLE_RATE
            )

            if len(speech_timestamps) == 0:

                print("\n[SEM FALA DETECTADA]")

                continue

            max_val = np.max(np.abs(audio_np))

            if max_val > 0:
                audio_np = audio_np / max_val

            print("\n===================================================")
            print("NOVO CHUNK PROCESSADO")

            # ==========================================
            # TRANSCRIÇÃO
            # ==========================================

            print("\n[TRANSCRIÇÃO]")

            segments, info = whisper_model.transcribe(
                audio_np,
                language="pt"
            )

            full_text = ""

            for segment in segments:
                full_text += segment.text + " "

            full_text = full_text.strip()

            print("Texto:", full_text)

            # ==========================================
            # EMOÇÃO
            # ==========================================

            print("\n[EMOÇÃO DETECTADA]")

            result = emotion_classifier(
                audio_np,
                sampling_rate=16000
            )

            top_emotion = max(
                result,
                key=lambda x: x["score"]
            )

            emotion_label = top_emotion["label"].lower()
            emotion_score = top_emotion["score"]

            print(
                f"Emoção principal: {emotion_label}"
            )

            print(
                f"Confiança: {emotion_score:.2f}"
            )

            # ==========================================
            # ANÁLISE VOCAL
            # ==========================================

            print("\n[ANÁLISE VOCAL]")

            features = smile.process_signal(
                audio_np,
                SAMPLE_RATE
            )

            loudness = None
            pitch = None
            mfcc1 = None
            alpha_ratio = None

            # ==========================================
            # EXTRAÇÃO
            # ==========================================

            if "loudness_sma3_amean" in features.columns:

                loudness = features[
                    "loudness_sma3_amean"
                ].values[0]

            if "F0semitoneFrom27.5Hz_sma3nz_amean" in features.columns:

                pitch = features[
                    "F0semitoneFrom27.5Hz_sma3nz_amean"
                ].values[0]

            if "mfcc1_sma3_amean" in features.columns:

                mfcc1 = features[
                    "mfcc1_sma3_amean"
                ].values[0]

            if "alphaRatioV_sma3nz_amean" in features.columns:

                alpha_ratio = features[
                    "alphaRatioV_sma3nz_amean"
                ].values[0]

            # ==========================================
            # ENERGIA VOCAL
            # ==========================================

            if loudness is not None:

                if loudness < 0.05:

                    print(
                        "A voz demonstra energia extremamente baixa."
                    )

                elif loudness < 0.15:

                    print(
                        "A voz aparenta baixa energia."
                    )

                elif loudness > 0.5:

                    print(
                        "A voz demonstra alta energia e intensidade."
                    )

                else:

                    print(
                        "Energia vocal moderada."
                    )

            # ==========================================
            # TOM DE VOZ
            # ==========================================

            if pitch is not None:

                if pitch < 20:

                    print(
                        "Tom de voz mais grave e monótono."
                    )

                elif pitch > 35:

                    print(
                        "Tom de voz mais agudo e expressivo."
                    )

                else:

                    print(
                        "Tom de voz equilibrado."
                    )

            # ==========================================
            # EXPRESSIVIDADE
            # ==========================================

            if mfcc1 is not None:

                if mfcc1 < 0:

                    print(
                        "A fala aparenta menor expressividade emocional."
                    )

                elif mfcc1 > 20:

                    print(
                        "A fala aparenta alta expressividade."
                    )

                else:

                    print(
                        "Expressividade vocal moderada."
                    )

            # ==========================================
            # CLAREZA VOCAL
            # ==========================================

            if alpha_ratio is not None:

                if alpha_ratio < -20:

                    print(
                        "A voz aparenta estar abafada ou cansada."
                    )

                elif alpha_ratio > -5:

                    print(
                        "A voz aparenta estar clara e projetada."
                    )

                else:

                    print(
                        "Clareza vocal dentro do esperado."
                    )

            # ==========================================
            # INTERPRETAÇÃO EMOCIONAL
            # ==========================================

            print("\n[INTERPRETAÇÃO EMOCIONAL]")

            if emotion_label == "sad":

                if emotion_score > 0.6:

                    print(
                        "Forte indício emocional de tristeza."
                    )

                else:

                    print(
                        "Leve tendência emocional à tristeza."
                    )

            elif emotion_label == "fearful":

                if emotion_score > 0.5:

                    print(
                        "Possível ansiedade ou tensão emocional."
                    )

            elif emotion_label == "angry":

                if emotion_score > 0.5:

                    print(
                        "Possível irritabilidade ou frustração."
                    )

            elif emotion_label == "happy":

                if emotion_score > 0.5:

                    print(
                        "Tom emocional positivo."
                    )

            elif emotion_label == "neutral":

                print(
                    "Tom emocional neutro."
                )

            # ==========================================
            # ANÁLISE COMBINADA
            # ==========================================

            print("\n[ANÁLISE COMBINADA]")

            if (
                emotion_label == "sad"
                and loudness is not None
                and loudness < 0.15
            ):

                print(
                    "Possível combinação de tristeza e baixa energia vocal."
                )

            if (
                emotion_label == "fearful"
                and pitch is not None
                and pitch > 30
            ):

                print(
                    "Possível tensão emocional detectada."
                )

            if (
                emotion_label == "happy"
                and loudness is not None
                and loudness > 0.3
            ):

                print(
                    "A voz demonstra energia positiva."
                )

            # ==========================================
            # ANÁLISE DA TRANSCRIÇÃO
            # ==========================================

            print("\n[ANÁLISE DA FALA]")

            texto_lower = full_text.lower()

            negative_words = [
                "cansado",
                "triste",
                "desanimado",
                "ansioso",
                "deprimido",
                "sozinho",
                "medo",
                "morrer",
            ]

            positive_words = [
                "feliz",
                "animado",
                "ótimo",
                "excelente",
                "empolgado",
            ]

            negative_count = sum(
                1 for word in negative_words
                if word in texto_lower
            )

            positive_count = sum(
                1 for word in positive_words
                if word in texto_lower
            )

            if negative_count >= 2:

                print(
                    "A fala possui sinais linguísticos negativos."
                )

            if positive_count >= 2:

                print(
                    "A fala possui sinais linguísticos positivos."
                )

            if (
                negative_count == 0
                and positive_count == 0
            ):

                print(
                    "Nenhum padrão linguístico relevante detectado."
                )

            # ==========================================
            # SCORE FINAL
            # ==========================================

            print("\n[RESULTADO FINAL]")

            risk_score = 0

            if emotion_label == "sad":
                risk_score += 2

            if emotion_label == "fearful":
                risk_score += 1

            if loudness is not None and loudness < 0.1:
                risk_score += 1

            if negative_count >= 2:
                risk_score += 2

            print(
                f"Score emocional: {risk_score}/5"
            )

            if risk_score >= 4:

                print(
                    "Possíveis indícios emocionais relevantes."
                )

            elif risk_score >= 2:

                print(
                    "Leves sinais emocionais detectados."
                )

            else:

                print(
                    "Sem sinais emocionais relevantes."
                )

        except Exception as e:

            print("Erro:", e)

# ============================================================
# THREAD PROCESSAMENTO
# ============================================================

In [10]:
processing_thread = threading.Thread(
    target=process_audio,
    daemon=True
)

processing_thread.start()

# ============================================================
# START MICROFONE
# ============================================================

In [11]:
print("\n===================================================")
print("🎤 Ouvindo microfone em tempo real...")
print("Pressione CTRL+C para parar.")
print("===================================================\n")

try:

    with sd.InputStream(
        samplerate=SAMPLE_RATE,
        channels=CHANNELS,
        callback=audio_callback,
    ):

        while True:
            time.sleep(0.1)

except KeyboardInterrupt:

    print("\nEncerrado.")


🎤 Ouvindo microfone em tempo real...
Pressione CTRL+C para parar.


NOVO CHUNK PROCESSADO

[TRANSCRIÇÃO]
Texto: Olá, tudo bem? O meu nome é Igor. Aí eu tô meio triste.

[EMOÇÃO DETECTADA]
Emoção principal: angry
Confiança: 0.13

[ANÁLISE VOCAL]
A voz demonstra alta energia e intensidade.
Tom de voz equilibrado.
Expressividade vocal moderada.
Clareza vocal dentro do esperado.

[INTERPRETAÇÃO EMOCIONAL]

[ANÁLISE COMBINADA]

[ANÁLISE DA FALA]

[RESULTADO FINAL]
Score emocional: 0/5
Sem sinais emocionais relevantes.

[SEM FALA DETECTADA]

[SEM FALA DETECTADA]

Encerrado.
